# ==========================================================
# 05_methods.ipynb - Tree Decomposition & Diagnostics
# ==========================================================

In [ ]:
import pathlib
import numpy as np
import pandas as pd

PROJECT_ROOT = pathlib.Path.cwd().parent
CLEAN_DIR = PROJECT_ROOT / "data" / "clean"

df_bookings = pd.read_csv(CLEAN_DIR / "bookings_clean.csv.gz", parse_dates=["slot_at"])

In [ ]:
# Define H1 periods
h1_2025_mask = df_bookings["slot_at"].between("2025-01-01", "2025-06-30 23:59:59")
h1_2026_mask = df_bookings["slot_at"].between("2026-01-01", "2026-06-30 23:59:59")

df_bookings["half"] = np.where(h1_2025_mask, "H1_2025", np.where(h1_2026_mask, "H1_2026", "other"))
df_h1 = df_bookings[df_bookings["half"] != "other"].copy()

# Determine Patient ID column
person_col = "patient_id"

In [ ]:
# ----------------------------------------------------------
# Step 1: Build the Performance Metric Tree
# ----------------------------------------------------------
def calculate_tree_metrics(df, group_cols):
    grouped = df.groupby(group_cols + ["half"]).agg(
        tests=("booking_id", "size"),
        visits=("booking_id", "nunique"),
        people=(person_col, "nunique")
    )
    grouped["visits_per_person"] = grouped["visits"] / grouped["people"]
    grouped["tests_per_visit"] = grouped["tests"] / grouped["visits"]
    return grouped

df_network_tree = calculate_tree_metrics(df_h1, group_cols=[])
print("--- Network Performance Tree ---")
print(df_network_tree.round(3))

In [ ]:
# ----------------------------------------------------------
# Step 2: City x Channel Performance Breakdown
# ----------------------------------------------------------
df_city_channel = df_h1.groupby(["city", "channel", "half"]).size().unstack("half").fillna(0)
df_city_channel["growth_pct"] = ((df_city_channel["H1_2026"] / df_city_channel["H1_2025"]) - 1) * 100

print("\n--- Growth by City and Channel ---")
print(df_city_channel.round(1))